## Data Preprocessing

### Import Necessary Libraries

In [2]:
import sys
sys.path.append('../src')
sys.path.append('../src/data')

import random
import re

import numpy as np
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration

from data_loader import DataLoader, DataLoaderSelector
from augment_dataset import (
    add_synthetic_conclusions_straightforward,
    add_synthetic_conclusion_from_temps,
    add_enhanced_synthetic_conclusions,
    preprocess_conclusion
)

### Load and Clean Dataset

In [3]:
mwr = DataLoaderSelector(data_path='data_th_scale.csv')

### Access the Dataframe

In [4]:
mwr_df = mwr.data_loader.data.copy(deep=True)  # full deep copy of raw data

In [5]:
print(mwr_df['r:Th'].value_counts(normalize=True))

r:Th
2    0.294479
1    0.289074
0    0.241170
4    0.080335
5    0.064945
3    0.029997
Name: proportion, dtype: float64


## Generate Synthetic Descriptions

### Straightforward Conclusions

In [5]:
# Th: Thermal asymmetry with:
# 0: no changes
# 1: slightly elevated temperature
# 2: moderately elevated temperature (surface)
# 3: moderately elevated temperature (surface and depth)
# 4: increased temperature (surface and depth), partial asymmetry
# 5: increased temperature (surface and depth), greater and clear asymmetry

In [6]:
# add the straightforward conclusions
mwr_df_simple = add_synthetic_conclusions_straightforward(mwr_df.copy(deep=True))

In [7]:
mwr_df_simple.head()

,Examination ID,Conclusion,r:Th,Weight,Height,Ambient temperature,r:AgeInYears,Mammary diameter,Cycle,Day from the first day,...,R8 sk,L8 sk,R9 sk,L9 sk,T1 sk,T2 sk,R0 sk,L0 sk,Conclusion (Tr),Synthetic_Conclusion
0,00002A00010B1,temperatura v v predelu mastektomije je nekoli...,0,0,0,22,43,17,28,16,...,32.7,31.4,33.8,32.6,33.1,32.9,33.3,31.6,The temperature in the area of the mastectomy ...,No thermal changes detected.
1,00002A00010C1,v desni dojki je temperatura lepo razporejena ...,0,0,0,22,45,20,30,9,...,32.8,31.5,33.2,32.6,32.5,33.2,33.3,31.4,"In the right breast, the temperature is well d...",No thermal changes detected.
2,00002A00023C1,"1/ razporeditev temperature je asimetrična, ve...",1,0,0,21,53,25,0,-1,...,30.9,30.0,30.5,30.6,31.6,31.8,30.9,31.6,"1/ The temperature distribution is asymmetric,...",Slightly elevated temperature.
3,00002A00037C1,temperatura v obeh dojkah je pravilno porazdel...,0,0,0,25,54,24,0,-1,...,32.0,31.2,32.3,33.0,32.7,32.6,33.4,33.5,The temperature in both breasts is evenly dist...,No thermal changes detected.
4,00002A00037D1,"dojki sta srednje veliki, mehki, rahlo vozliča...",0,0,0,23,55,25,0,-1,...,31.4,31.7,33.3,32.5,33.6,33.4,33.7,33.1,"The breasts are medium-sized, soft, slightly n...",No thermal changes detected.


In [8]:
mwr_df_simple['y_binary'] = mwr_df_simple['r:Th'].apply(lambda x: 0 if x == 0 else 1)
y_class = mwr_df_simple['y_binary']

In [9]:
mwr_df_simple.to_csv('mwr_simple.csv', index=False)

### Temperature Based Conclusions

In [10]:
mwr_df_temps = add_synthetic_conclusion_from_temps(mwr_df.copy(deep=True))

In [11]:
print(mwr_df_temps['Synthetic_Conclusion'])

0        No thermal changes detected. Internal temps ex...
1        No thermal changes detected. Internal temps ex...
2        Very mild regional irregularities observed. Sl...
3                             No thermal changes detected.
4                             No thermal changes detected.
                               ...                        
24247    Significant thermal irregularities observed. M...
24248    Significant thermal irregularities observed. A...
24249                         No thermal changes detected.
24250                 Low-grade thermal anomalies present.
24251    Low-grade thermal anomalies present. Internal ...
Name: Synthetic_Conclusion, Length: 24236, dtype: object


In [12]:
mwr_df_temps.head()

,Examination ID,Conclusion,r:Th,Weight,Height,Ambient temperature,r:AgeInYears,Mammary diameter,Cycle,Day from the first day,...,R8 sk,L8 sk,R9 sk,L9 sk,T1 sk,T2 sk,R0 sk,L0 sk,Conclusion (Tr),Synthetic_Conclusion
0,00002A00010B1,temperatura v v predelu mastektomije je nekoli...,0,0,0,22,43,17,28,16,...,32.7,31.4,33.8,32.6,33.1,32.9,33.3,31.6,The temperature in the area of the mastectomy ...,No thermal changes detected. Internal temps ex...
1,00002A00010C1,v desni dojki je temperatura lepo razporejena ...,0,0,0,22,45,20,30,9,...,32.8,31.5,33.2,32.6,32.5,33.2,33.3,31.4,"In the right breast, the temperature is well d...",No thermal changes detected. Internal temps ex...
2,00002A00023C1,"1/ razporeditev temperature je asimetrična, ve...",1,0,0,21,53,25,0,-1,...,30.9,30.0,30.5,30.6,31.6,31.8,30.9,31.6,"1/ The temperature distribution is asymmetric,...",Very mild regional irregularities observed. Sl...
3,00002A00037C1,temperatura v obeh dojkah je pravilno porazdel...,0,0,0,25,54,24,0,-1,...,32.0,31.2,32.3,33.0,32.7,32.6,33.4,33.5,The temperature in both breasts is evenly dist...,No thermal changes detected.
4,00002A00037D1,"dojki sta srednje veliki, mehki, rahlo vozliča...",0,0,0,23,55,25,0,-1,...,31.4,31.7,33.3,32.5,33.6,33.4,33.7,33.1,"The breasts are medium-sized, soft, slightly n...",No thermal changes detected.


In [15]:
mwr_df_temps['y_binary'] = mwr_df_temps['r:Th'].apply(lambda x: 0 if x == 0 else 1)
y_class = mwr_df_temps['y_binary']

In [16]:
mwr_df_temps.to_csv('mwr_temps.csv', index=False)

### More Advanced Descriptions

In [7]:
mwr_df_adv = add_enhanced_synthetic_conclusions(mwr_df.copy(deep=True))

In [9]:
print(mwr_df_adv['Enhanced_Synthetic_Conclusion'])

0                                                                                                                                                                                                                                                             The temperature distribution shows physiological asymmetry within normal parameters. The difference between deep and superficial values indicates normal tissue metabolism. There are no signs of local hyperthermia or abnormal thermal activity. I advise regular self-examination of the breasts once a month and a systematic examination once a year.
1                                                                                                                                                                                                                                                              The temperature distribution shows physiological asymmetry within normal parameters. The difference between deep and superficial values indicate

In [20]:
mwr_df_adv['y_binary'] = mwr_df_adv['r:Th'].apply(lambda x: 0 if x == 0 else 1)
y_class = mwr_df_adv['y_binary']

In [21]:
mwr_df_adv.to_csv('mwr_adv.csv', index=False)

### Real Descriptions

In [8]:
# Remove column width truncation
pd.set_option('display.max_colwidth', None)

# Now print
print(mwr_df['Conclusion (Tr)'].head(30))

0                                                                                                                                                         The temperature in the area of the mastectomy is slightly higher than in the area of the left breast, but the difference between the deep and superficial values indicates rest or absence of proliferation. In the left breast, at the border of the lower quadrants, there is a typical hardening approximately 5mm in size, which is not thermally active. I recommend monitoring at one-month intervals. Otherwise, a follow-up in 6 months.
1                                                                                                                                                In the right breast, the temperature is well distributed and does not deviate from the average. In the left breast, a cooler area is visible around the nipple, specifically 0.9 degrees, which is still within normal limits. The left breast is also quite nodular, partic

In [23]:
mwr_df['Conclusion (Tr)'] = mwr_df['Conclusion (Tr)'].apply(preprocess_conclusion)

In [24]:
mwr_df.shape[0]

24236

In [25]:
# remove rows with empty strings
mwr_df = mwr_df[mwr_df['Conclusion (Tr)'] != ""].copy()
mwr_df.shape[0]

24162

In [26]:
print(mwr_df['Conclusion (Tr)'].head(30))

0                                                                                                                                                      The temperature in the area of the mastectomy is slightly higher than in the area of the left breast, but the difference between the deep and superficial values indicates rest or absence of proliferation. In the left breast, at the border of the lower quadrants, there is a typical hardening approximately 5mm in size, which is not thermally active. I recommend monitoring at one-month intervals. Otherwise, a follow-up in 6 months.
1                                                                                                                                             In the right breast, the temperature is well distributed and does not deviate from the average. In the left breast, a cooler area is visible around the nipple, specifically 0.9 degrees, which is still within normal limits. The left breast is also quite nodular, particularly

In [27]:
# check all values are strings
non_string_count = mwr_df['Conclusion (Tr)'].apply(lambda x: not isinstance(x, str)).sum()

print("Number of non-string values:", non_string_count)

Number of non-string values: 0


### Binary encoding

In [28]:
mwr_df['y_binary'] = mwr_df['r:Th'].apply(lambda x: 0 if x == 0 else 1)
y_class = mwr_df['y_binary']

In [29]:
mwr_df.to_csv('mwr.csv', index=False)